# Demonstration of the `gsf` package


This notebook runs **GalacticStructureFinder** on the bundled example galaxy (`tests/sim1`) for the simplest case of **two kinematic components**, and shows the result: the face-on/edge-on moment maps of the named components, and a table of their physical properties.

The first step computes the gravitational potential at every stellar particle by direct two-body summation, which is the slowest part of the run; exporting `OMP_NUM_THREADS` (up to the number of cores on your machine) before starting Jupyter speeds it up considerably.


In [ ]:
import os, glob, pickle
import numpy as np
from IPython.display import Image, HTML, display
import gsf

# Locate the bundled example galaxy (works whether this notebook runs from the
# repository root or from the notebooks/ directory).
sim_dir = next(p for p in ('tests/sim1', '../tests/sim1', '../../tests/sim1')
               if os.path.exists(os.path.join(p, 'sim1.halo_1.star.dat')))
star = os.path.join(sim_dir, 'sim1.halo_1.star.dat')
gas  = os.path.join(sim_dir, 'sim1.halo_1.gas.dat')
dark = os.path.join(sim_dir, 'sim1.halo_1.dark.dat')

out_dir = 'demo_output/'
os.makedirs(out_dir, exist_ok=True)

# Decompose the galaxy into two components. This runs the whole pipeline:
# potential -> derived quantities -> GMM clustering -> moment maps -> naming.
gsf.gsf(star, gas, dark, number_of_clusters=2, out_dir=out_dir, plot=False, verbose=False)


## Component maps

For every component `gsf` produces, from left to right, the face-on surface mass density, the edge-on surface mass density, and the edge-on line-of-sight velocity. The rows are stacked into a single mosaic (the whole galaxy on top, then the components ordered by mass fraction), with each component labelled by the physical name assigned by `tag_components`.


In [ ]:
# The two-inclination mosaic written by tag_components -> plot_2inclinations_moment_maps.
mosaic = sorted(glob.glob(out_dir + '*_2inc_mosaic.png'))[0]
display(Image(filename=mosaic))


## Component properties

For each component, `gsf` writes the summary table of physical properties as a ready-to-`\input` LaTeX file (`*table.tex`). Below we parse that file and render it as an HTML table, so the notebook shows exactly the same numbers as the published table: mass and mass fraction, the equatorial-plane asymmetry `b/a`, the flatness and rotational-support parameters, velocity dispersions, and the mass-weighted medians (with 16th/84th percentiles) of radius, rotational velocity, age, metallicity, circularities, and normalized binding energy.

In [ ]:
import re
from IPython.display import HTML

tex = open(glob.glob(out_dir + '*table.tex')[0]).read()
body = tex.split(r'\begin{tabular}', 1)[1].split(r'\end{tabular}', 1)[0]
body = body.split('}', 1)[1]                      # drop the {cccc} column spec

def _tex_to_html(cell):
    # median^{+hi}_{-lo}  ->  value with superscript / subscript errors
    cell = re.sub(r'\$\^\{\\rm \+([^}]*)\}_\{\\rm -([^}]*)\}\$',
                  r'<sup>+\1</sup><sub>-\2</sub>', cell)
    cell = re.sub(r'\$\^\{\\rm ([^}]*)\}\$', r'<sup>\1</sup>', cell)   # e.g. 10^{10}
    cell = cell.replace('$', '')
    for a, b in (
        (r'\frac{M}{M_{\rm *}}', 'M/M<sub>*</sub>'),
        (r'_{\rm\odot}', '<sub>\u2609</sub>'),
        (r'v_{\rm\phi}', 'v<sub>\u03c6</sub>'),
        (r'\rm\sigma', '\u03c3'), (r'\sigma', '\u03c3'),
        (r'\kappa', '\u03ba'), (r'\xi', '\u03be'),
        (r'\phi', '\u03c6'), (r'\odot', '\u2609'), (r'\rm', ''),
    ):
        cell = cell.replace(a, b)
    return cell.replace('{', '').replace('}', '').replace('\\', '').strip()

rows = []
for raw in body.split(r'\\'):
    raw = raw.replace(r'\hline', '').strip()
    if raw:
        rows.append([_tex_to_html(c) for c in raw.split('&')])
header, data = rows[0], rows[1:]

th = "style='padding:4px 12px;border-bottom:2px solid #444'"
td = "style='padding:4px 12px;text-align:center'"
tl = "style='padding:4px 12px;text-align:left'"
html = "<table style='border-collapse:collapse;font-family:sans-serif'>"
html += "<tr>" + "".join("<th %s>%s</th>" % (th, h) for h in header) + "</tr>"
for r in data:
    html += ("<tr><td %s><b>%s</b></td>" % (tl, r[0])
             + "".join("<td %s>%s</td>" % (td, c) for c in r[1:]) + "</tr>")
html += "</table>"
HTML(html)


## The same table as raw LaTeX

The `*table.tex` file is ready to `\input` directly into a paper; its contents:

In [ ]:
tex = glob.glob(out_dir + '*table.tex')[0]
print('Written to:', tex, '\n')
print(open(tex).read())
